# Phase 5: Two-Year Weather-Only Forecast Residuals

This notebook joins the certified deterministic forecast panel to the
certified Hong Kong Observatory daily maximum-temperature observations.

The residual is defined by

\[
R_{d,r}
=
T_d^{\mathrm{HKO}}
-
\widehat T_{d,r}^{\mathrm{det}}.
\]

A positive residual therefore means that the deterministic forecast was
lower than the realised HKO daily maximum.

This phase does not access market prices, fit a model, select a model,
calibrate probabilities or calculate trading returns.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()

for candidate in (ROOT, *ROOT.parents):
    if (
        candidate
        / "config/v2/"
        "weather_only_residual_spec.json"
    ).exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Repository root not found."
    )

completed = subprocess.run(
    [
        sys.executable,
        str(
            ROOT
            / "tools/v2/"
            "build_weather_only_residual_panel.py"
        ),
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(
        "Phase 5 residual construction failed."
    )


PHASE 5 WEATHER-ONLY RESIDUAL PANEL
Status: TWO_YEAR_FORECAST_RESIDUAL_PANEL_CERTIFIED
Forecast source column: forecast_daily_max_c
Weather-only training dates: 730
Decision rules: 4
Residual rows: 2920
Training interval: 2024-03-16 to 2026-03-15
Mean residual: 1.429658 degrees Celsius
Residual standard deviation: 1.473250
Mean absolute error: 1.680068
Root mean squared error: 2.052716
Missing forecasts: 0
Missing observations: 0
Market prices accessed: False
Polymarket outcomes accessed: False
Model fitted or selected: False
PHASE 5 CORE CONSTRUCTION: PASSED



In [2]:
import json
import pandas as pd

manifest = json.loads(
    (
        ROOT
        / "data/manifests/v2/"
        "05_weather_only_residual_manifest.json"
    ).read_text(encoding="utf-8")
)

panel = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "05_weather_only_forecast_residual_panel.csv"
)

rule_summary = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "05_weather_only_residual_rule_summary.csv"
)

print("Status:", manifest["status"])
print("Residual rows:", len(panel))
print("Dates:", panel["target_date"].nunique())
print("Decision rules:", panel["decision_rule"].nunique())
print()
print(rule_summary.to_string(index=False))

Status: TWO_YEAR_FORECAST_RESIDUAL_PANEL_CERTIFIED
Residual rows: 2920
Dates: 730
Decision rules: 4

 decision_rule  decision_rule_order  rows  dates  mean_forecast_daily_max_c  mean_hko_daily_max_c  mean_residual_c  median_residual_c  residual_standard_deviation_c  mean_absolute_error_c  root_mean_squared_error_c  minimum_residual_c  maximum_residual_c
     24h_prior                    1   730    730                  25.835616                 27.28         1.444384                1.4                       1.549097               1.735890                   2.117229                -4.0                 7.6
     12h_prior                    2   730    730                  25.857260                 27.28         1.422740                1.4                       1.478183               1.674247                   2.050907                -3.0                 6.5
      6h_prior                    3   730    730                  25.846438                 27.28         1.433562                1.5 

In [3]:
import numpy as np

assert len(panel) == 2920
assert panel["target_date"].nunique() == 730
assert panel["decision_rule"].nunique() == 4

assert not panel.duplicated(
    ["target_date", "decision_rule"]
).any()

assert np.allclose(
    panel["residual_c"],
    (
        panel["hko_daily_max_c"]
        - panel["forecast_daily_max_c"]
    ),
    atol=1e-12,
    rtol=0.0,
)

assert manifest["market_prices_accessed"] is False
assert manifest["polymarket_outcomes_accessed"] is False
assert manifest["model_fitted"] is False
assert manifest["model_selected"] is False

print("PHASE 5 NOTEBOOK VERIFICATION: PASSED")

PHASE 5 NOTEBOOK VERIFICATION: PASSED
